# Detecting exercise editor activity from simulated keystroke counts

This offline experiment tests whether aggregate editor activity can identify windows that the simulator labels as exercise editing. The data consists entirely of simulated counts. The model reads no text, key identities, or exact event timestamps. The two-minute observation window also misses activity away from the editor.

See [research notes](KEYSTROKE-RESEARCH.md) for the distinction between editing activity and evidence of progress on an exercise.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit

from keystroke_data import (
    BIN_SECONDS, FEATURE_NAMES, SEED, WINDOW_BINS, simulate_windows, window_features
)
from keystroke_model import (
    MODEL_PATH, PREDICTION_THRESHOLD, build_model, export_model, score_windows
)

windows = simulate_windows()
x = np.stack([window_features(window.counts) for window in windows])
y = np.array([window.target for window in windows])
groups = np.array([window.profile for window in windows])
scenarios = np.array([window.scenario for window in windows])
split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train, holdout = next(split.split(x, y, groups))
assert set(groups[train]).isdisjoint(groups[holdout])

print(json.dumps({
    "seed": SEED,
    "windows": len(windows),
    "profiles": len(set(groups)),
    "window_seconds": BIN_SECONDS * WINDOW_BINS,
    "training_profiles": len(set(groups[train])),
    "holdout_profiles": len(set(groups[holdout])),
    "holdout_windows": len(holdout),
    "holdout_positives": int(y[holdout].sum()),
    "features": FEATURE_NAMES,
}, indent=2))

## Hold out whole synthetic profiles

Profiles vary in typical editing speed and correction rate. Each profile contributes all five scenarios. The positive label covers brief and sustained editing scenarios; the other scenarios cover reading or review, idle time, and paste with few edits. Scenario labels describe how the simulator generated a window.

Fit the scaler and classifier on the training profiles. Use a fixed threshold of 0.5 and compare with a rule that flags a window with edits in at least three bins.

In [ ]:
def metrics(truth, prediction):
    tn, fp, fn, tp = confusion_matrix(truth, prediction, labels=[0, 1]).ravel()
    return {
        "precision": round(precision_score(truth, prediction, zero_division=0), 3),
        "recall": round(recall_score(truth, prediction, zero_division=0), 3),
        "f1": round(f1_score(truth, prediction, zero_division=0), 3),
        "balanced_accuracy": round(balanced_accuracy_score(truth, prediction), 3),
        "true_positive": int(tp), "false_positive": int(fp),
        "true_negative": int(tn), "false_negative": int(fn),
    }


model = build_model().fit(x[train], y[train])
scores = model.predict_proba(x[holdout])[:, 1]
prediction = scores >= PREDICTION_THRESHOLD
activity_rule = x[holdout, FEATURE_NAMES.index("active_edit_bins")] >= 3

print(json.dumps({
    "classifier": {
        **metrics(y[holdout], prediction),
        "average_precision": round(average_precision_score(y[holdout], scores), 3),
    },
    "three_active_bins_rule": metrics(y[holdout], activity_rule),
}, indent=2))

for scenario in sorted(set(scenarios[holdout])):
    rows = scenarios[holdout] == scenario
    print(f"{scenario}: {int(prediction[rows].sum())}/{int(rows.sum())} flagged")

fig, ax = plt.subplots(figsize=(5.2, 4))
ConfusionMatrixDisplay.from_predictions(
    y[holdout], prediction, display_labels=["Other", "Exercise editing"],
    colorbar=False, ax=ax,
)
ax.set_title("Synthetic profiles held out")
fig.tight_layout()
plt.show()

## Check where identical traces hide different activity

Give the model identical high-activity counts for editing an exercise and typing unrelated material in the same editor. Give it identical empty counts for thinking through a solution away from the editor and leaving the desk. Each pair must produce the same score because the model observes only the count features.

In [ ]:
example_windows = simulate_windows(profiles=3, windows_per_scenario=1, seed=99)
busy = max(
    (window for window in example_windows if window.scenario == "sustained_editing"),
    key=lambda window: window.counts[:, 0].sum(),
).counts
quiet = np.zeros_like(busy)
cases = {
    "editing_exercise": busy,
    "typing_unrelated_material": busy.copy(),
    "thinking_away_from_editor": quiet,
    "away_from_desk": quiet.copy(),
}
examples = np.stack([window_features(trace) for trace in cases.values()])
example_scores = model.predict_proba(examples)[:, 1]
assert example_scores[0] == example_scores[1]
assert example_scores[2] == example_scores[3]
print(json.dumps({
    name: {"score": round(float(score), 4), "flagged": bool(score >= PREDICTION_THRESHOLD)}
    for name, score in zip(cases, example_scores, strict=True)
}, indent=2))

In [ ]:
export_model(model)
reloaded, threshold = score_windows(x[holdout])
np.testing.assert_allclose(reloaded, scores, atol=1e-12, rtol=0)
assert threshold == PREDICTION_THRESHOLD
print(f"Exported {MODEL_PATH.name} ({MODEL_PATH.stat().st_size:,} bytes).")
print(f"Reloaded artifact reproduced all {len(holdout)} held-out scores.")